# RO from a PFC

Takes a Price Forward Curve for power / gas / EUA, works out which target quarters it
implies, and computes the RO for each of them as of a close-of-business date.

The curve here is `data/example_pfc_daily_baseload_2026-08-26.csv`: the daily baseload
average of the hourly `shaped_curve_wide_hourly_2026-08-26.csv`, one row per delivery day
and one column per commodity. `pfc_average` takes an unweighted mean of the rows inside a
delivery window, so a daily baseload curve gives the same RO as the hourly one it was built
from.

The PFC only supplies the **floating** half of each leg: the part of a regulatory averaging
window that has not been observed yet as of `COB_DATE`. The already-observed part still comes
from `data/futures.csv`.

In [1]:
import os
from datetime import date
from pathlib import Path

import pandas as pd

DATA = (Path("..") / "data").resolve()

# Market data (futures.csv, the PFC, holidays.csv) is read from the data/ folder of the
# current working directory. A notebook runs in notebooks/, so point RO_DATA_DIR at ../data.
os.environ["RO_DATA_DIR"] = str(DATA)

from ro_calculation import delivery_window, load_pfc, load_price_frames, run_asof

pd.set_option("display.float_format", "{:.2f}".format)

COB_DATE = date(2026, 8, 26)
PFC_PATH = DATA / "example_pfc_daily_baseload_2026-08-26.csv"
FUTURES_PATH = DATA / "futures.csv"

## 1. Load the PFC

`load_pfc` drops the exchange suffix from the curve columns (`power_EEX` -> `power`), which
is how `gather_price_components` recognises them. The delivery days are the `datetime`
column: one row per day here, one row per hour in the hourly curve.

In [2]:
pfc = load_pfc(PFC_PATH)
delivery_days = pfc["datetime"].dt.date

print(f"{len(pfc):,} rows, {delivery_days.min()} -> {delivery_days.max()}")
print("columns:", pfc.columns.tolist())
pfc.head()

2,191 rows, 2026-01-01 -> 2031-12-31
columns: ['datetime', 'EUA', 'gas', 'power', 'shape_type']


,datetime,EUA,gas,power,shape_type
0,2026-01-01,85.24,28.86,81.87,realised
1,2026-01-02,86.22,29.32,82.94,realised
2,2026-01-03,86.22,30.54,88.43,realised
3,2026-01-04,86.22,31.54,66.78,realised
4,2026-01-05,85.17,29.03,95.44,realised


## 2. Which quarters the PFC implies

Every quarter the delivery days touch is a target quarter. The curve defines the scope: if a
day of a quarter is in the file, that quarter gets an RO.

`COB_DATE` does not filter this list. It decides only how much of each averaging window has
already been observed (section 4). A quarter whose delivery is entirely in the past is still
in scope and comes back as its published BOE value - `run_asof` checks
`data/ro_realised.xlsx` first and short-circuits to the resolution when one exists, without
touching the curve.

`covered` records whether the PFC spans every delivery window the calculation averages over.
The widest is the `Y` leg, the whole calendar year of the target quarter, so a quarter whose
year is only partly in the file is still priced, but off a truncated average.

In [3]:
def quarter_code(day: date) -> str:
    """Delivery day -> the target_quarter code it belongs to, e.g. 'q3_26'."""
    return f"q{(day.month - 1) // 3 + 1}_{day.year % 100:02d}"


def required_pfc_span(target_quarter: str) -> tuple[date, date]:
    """Widest delivery window the quarter's legs are averaged over: the Y leg's year."""
    windows = [delivery_window(target_quarter, leg) for leg in ("Y", "Q", "M1", "M2", "M3")]
    return min(w[0] for w in windows), max(w[1] for w in windows)


pfc_start, pfc_end = delivery_days.min(), delivery_days.max()

quarters = pd.DataFrame(
    [
        {
            "quarter": q,
            "delivery_end": delivery_window(q, "Q")[1],
            "needs_pfc_from": required_pfc_span(q)[0],
            "needs_pfc_to": required_pfc_span(q)[1],
            "delivery_days_in_pfc": int(sum(quarter_code(d) == q for d in delivery_days)),
        }
        for q in dict.fromkeys(quarter_code(d) for d in delivery_days)
    ]
)
quarters["covered"] = (quarters["needs_pfc_from"] >= pfc_start) & (quarters["needs_pfc_to"] <= pfc_end)
quarters

,quarter,delivery_end,needs_pfc_from,needs_pfc_to,delivery_days_in_pfc,covered
0,q1_26,2026-03-31,2026-01-01,2026-12-31,90,True
1,q2_26,2026-06-30,2026-01-01,2026-12-31,91,True
2,q3_26,2026-09-30,2026-01-01,2026-12-31,92,True
3,q4_26,2026-12-31,2026-01-01,2026-12-31,92,True
4,q1_27,2027-03-31,2027-01-01,2027-12-31,90,True
5,q2_27,2027-06-30,2027-01-01,2027-12-31,91,True
6,q3_27,2027-09-30,2027-01-01,2027-12-31,92,True
7,q4_27,2027-12-31,2027-01-01,2027-12-31,92,True
8,q1_28,2028-03-31,2028-01-01,2028-12-31,91,True
9,q2_28,2028-06-30,2028-01-01,2028-12-31,91,True


In [4]:
TARGET_QUARTERS = quarters["quarter"].tolist()

print(f"{len(TARGET_QUARTERS)} target quarters: {TARGET_QUARTERS[0]} -> {TARGET_QUARTERS[-1]}")
partial = quarters.loc[~quarters["covered"], "quarter"].tolist()
if partial:
    print("PFC does not span the full Y-leg year for:", ", ".join(partial))

24 target quarters: q1_26 -> q4_31


## 3. Futures for the fixed half

`data/futures.csv` carries ICE's *rolling* front-month EUA symbols. On roll-boundary days
two symbols resolve to the same contract with different closes - on 2026-07-27 both
`/ECF<4>` and `/ECF<5>` map to Dec-26. `_decompose_product` refuses to guess between
conflicting closes and raises, which would block `q4_26` and `q1_27` (their EUA leg *is*
Dec-26). Keeping the higher-volume row resolves it. This is a workaround for a feed
artifact; the real fix belongs upstream in whatever produces `futures.csv`.

In [5]:
df_power, df_gas, df_eua = load_price_frames(FUTURES_PATH)

PRODUCT_DAY = ["delivery_date", "end_delivery_date", "market_date"]
conflicts = df_eua.groupby(PRODUCT_DAY)["close"].transform("nunique") > 1
print(f"EUA rows with a conflicting same-day close: {int(conflicts.sum())}")

df_eua = df_eua.sort_values("volume").drop_duplicates(subset=PRODUCT_DAY, keep="last")

EUA rows with a conflicting same-day close: 15


## 4. One RO per target quarter

`target_quarter` accepts a list, so this is a single fan-out call. Each quarter comes back
with a components frame of its own - one row per leg per commodity, a `total` row per
commodity, and a final `RO` row - and `ro` is those frames concatenated, every quarter
stacked into one long table. `ro[ro["commodity"] == "RO"]` is the per-quarter headline.

A realised quarter contributes its `RO` row alone: nothing is decomposed when the BOE
figure already exists.

The three price columns, per row:

- `fixed` - from the closes already observed. NaN while the leg's averaging window has
  not started, so it is empty for every leg dated past `COB_DATE`.
- `float` - the still-open part of the window, priced off this PFC.
- `reference` - the day-weighted blend of the two, and the figure to read. The `RO` row's
  `reference` is the RO itself.

`start_datetime` / `end_datetime` bound the quarter on the hourly grid, **hour-starting**:
the first hour of the quarter and the last one, so `q4_26` runs 2026-10-01 00:00 to
2026-12-31 23:00. The RO is constant across every hour in that range.

Quarters from 2029 on warn that `ro_parameters.xlsx` has no valores propios row for that
year and falls back to 2028's; their RO is priced off stale regulatory parameters.

In [6]:
results = run_asof(
    COB_DATE, df_power, df_gas, df_eua, TARGET_QUARTERS,
    use_pfc=True, pfc_csv_path=PFC_PATH,
)

ro = pd.concat([r["df"] for r in results.values()], ignore_index=True)

# Hour-starting: the first hour of the quarter, and the last one (23:00 of the final day),
# not the midnight that opens the next quarter.
bounds = {
    q: (
        pd.Timestamp(delivery_window(q, "Q")[0]),
        pd.Timestamp(delivery_window(q, "Q")[1]) + pd.Timedelta(hours=23),
    )
    for q in TARGET_QUARTERS
}
ro.insert(1, "start_datetime", ro["quarter"].map(lambda q: bounds[q][0]))
ro.insert(2, "end_datetime", ro["quarter"].map(lambda q: bounds[q][1]))
ro

C:\Users\R119752\OneDrive - Repsol\Escritorio\Coding\9. projects\3. RO calculation\src\ro_calculation\constants.py:199: UserWarning: ro_parameters.xlsx has no row for IT-01144 in 2029; using 2028 instead. The RO for that year is priced off stale valores propios — add the real row from the orden de parámetros retributivos.
  **installation_parameters(year, installation_type),
C:\Users\R119752\OneDrive - Repsol\Escritorio\Coding\9. projects\3. RO calculation\src\ro_calculation\constants.py:199: UserWarning: ro_parameters.xlsx has no row for IT-01144 in 2029; using 2028 instead. The RO for that year is priced off stale valores propios — add the real row from the orden de parámetros retributivos.
  **installation_parameters(year, installation_type),
C:\Users\R119752\OneDrive - Repsol\Escritorio\Coding\9. projects\3. RO calculation\src\ro_calculation\constants.py:199: UserWarning: ro_parameters.xlsx has no row for IT-01144 in 2029; using 2028 instead. The RO for that year is priced off stal

,quarter,start_datetime,end_datetime,commodity,strip,fixed,float,reference,pct_open,price_source,primary_direct
0,q1_26,2026-01-01,2026-03-31 23:00:00,RO,total,52.96,52.96,52.96,0.00,realised,NaN
1,q2_26,2026-04-01,2026-06-30 23:00:00,RO,total,89.49,89.49,89.49,0.00,realised,NaN
2,q3_26,2026-07-01,2026-09-30 23:00:00,RO,total,55.38,55.38,55.38,0.00,realised,NaN
3,q4_26,2026-10-01,2026-12-31 23:00:00,power,Y,61.09,NaN,61.09,0.00,[OMIP],True
4,q4_26,2026-10-01,2026-12-31 23:00:00,power,Q,104.89,118.10,108.97,0.31,"[OMIP, PFC]",False
...,...,...,...,...,...,...,...,...,...,...,...
292,q4_31,2031-10-01,2031-12-31 23:00:00,gas,M2,NaN,24.24,24.24,1.00,[PFC],False
293,q4_31,2031-10-01,2031-12-31 23:00:00,gas,M3,NaN,25.06,25.06,1.00,[PFC],False
294,q4_31,2031-10-01,2031-12-31 23:00:00,gas,total,NaN,23.99,23.99,1.00,[PFC],False
295,q4_31,2031-10-01,2031-12-31 23:00:00,EUA,total,NaN,99.14,99.14,1.00,[PFC],False


In [7]:
ro[ro['quarter'] == "q1_27"]

,quarter,start_datetime,end_datetime,commodity,strip,fixed,float,reference,pct_open,price_source,primary_direct
17,q1_27,2027-01-01,2027-03-31 23:00:00,power,Y,65.65,74.71,70.85,0.57,"[OMIP, PFC]",False
18,q1_27,2027-01-01,2027-03-31 23:00:00,power,Q,NaN,98.09,98.09,1.00,[PFC],False
19,q1_27,2027-01-01,2027-03-31 23:00:00,power,M1,NaN,115.53,115.53,1.00,[PFC],False
20,q1_27,2027-01-01,2027-03-31 23:00:00,power,M2,NaN,101.66,101.66,1.00,[PFC],False
21,q1_27,2027-01-01,2027-03-31 23:00:00,power,M3,NaN,77.43,77.43,1.00,[PFC],False
22,q1_27,2027-01-01,2027-03-31 23:00:00,power,total,NaN,92.30,91.33,0.89,"[OMIP, PFC]",False
23,q1_27,2027-01-01,2027-03-31 23:00:00,gas,Y,39.06,46.81,43.51,0.57,"[MIBGAS, PFC]",False
24,q1_27,2027-01-01,2027-03-31 23:00:00,gas,Q,NaN,62.36,62.36,1.00,[PFC],False
25,q1_27,2027-01-01,2027-03-31 23:00:00,gas,M1,NaN,64.53,64.53,1.00,[PFC],False
26,q1_27,2027-01-01,2027-03-31 23:00:00,gas,M2,NaN,63.50,63.50,1.00,[PFC],False


## 5. Data to be merged for spaguettis

In [8]:
# reference is the price to be in the end fed to PLEXOS through the spaguettis.
ro[ro['commodity'] == "RO"][['quarter', 'start_datetime', 'end_datetime', 'reference']]


,quarter,start_datetime,end_datetime,reference
0,q1_26,2026-01-01,2026-03-31 23:00:00,52.96
1,q2_26,2026-04-01,2026-06-30 23:00:00,89.49
2,q3_26,2026-07-01,2026-09-30 23:00:00,55.38
16,q4_26,2026-10-01,2026-12-31 23:00:00,56.74
30,q1_27,2027-01-01,2027-03-31 23:00:00,72.27
44,q2_27,2027-04-01,2027-06-30 23:00:00,81.76
58,q3_27,2027-07-01,2027-09-30 23:00:00,61.83
72,q4_27,2027-10-01,2027-12-31 23:00:00,60.78
86,q1_28,2028-01-01,2028-03-31 23:00:00,57.64
100,q2_28,2028-04-01,2028-06-30 23:00:00,65.26
